In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# scripts/fit.py

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from category_encoders import CatBoostEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from catboost import CatBoostRegressor
from sklearn.preprocessing import FunctionTransformer




from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine

In [4]:
load_dotenv()

True

In [ ]:
dst_host = os.environ.get('DB_DESTINATION_HOST')
dst_port = os.environ.get('DB_DESTINATION_PORT')
dst_username = os.environ.get('DB_DESTINATION_USER')
dst_password = os.environ.get('DB_DESTINATION_PASSWORD')
dst_db = os.environ.get('DB_DESTINATION_NAME')


In [6]:
dst_conn = create_engine(f'postgresql://{dst_username}:{dst_password}@{dst_host}:{dst_port}/{dst_db}')

In [42]:

SQL = """  SELECT * from flats_info"""
data = pd.read_sql(SQL, dst_conn)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90900 entries, 0 to 90899
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 90900 non-null  int64  
 1   building_id        90900 non-null  int64  
 2   build_year         90900 non-null  int64  
 3   building_type_int  90900 non-null  int64  
 4   latitude           90900 non-null  float64
 5   longitude          90900 non-null  float64
 6   ceiling_height     90900 non-null  float64
 7   flats_count        90900 non-null  int64  
 8   floors_total       90900 non-null  int64  
 9   has_elevator       90900 non-null  bool   
 10  flat_id            90900 non-null  int64  
 11  floor              90900 non-null  int64  
 12  kitchen_area       90900 non-null  float64
 13  living_area        90900 non-null  float64
 14  rooms              90900 non-null  int64  
 15  is_apartment       90900 non-null  bool   
 16  total_area         909

In [43]:
data = data.drop(columns=['floors_total', 'floor', 'rooms', 'build_year'])


In [59]:
target_col = ["price"]

id_col = [i for i in data.columns if i.endswith("id")]

bool_col = data.select_dtypes(include=bool).columns  

cat_ordinal_col = ["build_year_bin", "floor_bin", "floors_total_bin", 'rooms_bin']

cat_col = ["building_type_int"]

digital_notfilter = ["latitude", "longitude"]

digital_col = list(
    set(data.columns)
    - set(target_col)
    - set(id_col)
    - set(bool_col)
    - set(cat_ordinal_col)
    - set(cat_col)
    - set(digital_notfilter)
)
digital_col

['total_area', 'living_area', 'flats_count', 'kitchen_area', 'ceiling_height']

In [ ]:
def make_ordinary_col(data, cat_ordinal_col):
    ordinal_categories = ['low', 'med', 'hig']
    for col in cat_ordinal_col:
        data[col] = pd.Categorical(data[col], categories=ordinal_categories, ordered=True)

make_ordinary_col(data, cat_ordinal_col)

In [ ]:
def bool_to_int(x):
    return x.astype(int)
    
bool_to_int_fer = FunctionTransformer(bool_to_int, validate=False)

In [62]:
preprocessor = ColumnTransformer(
[
    ('binary', bool_to_int_fer, bool_col),
    ('cat', CatBoostEncoder(return_df=False), cat_ordinal_col+cat_col),
    ('num', StandardScaler(), digital_col)
],
remainder='drop',
verbose_feature_names_out=False
)

In [76]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error


In [ ]:
X = data.drop(target_col, axis=1)
y = data[target_col]

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,  verbose=0),
        ),
    ]
)

# Разделите данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Обучите модель
pipeline.fit(X_train, y_train)

# Сделайте предсказания
y_pred = pipeline.predict(X_test)


In [79]:

# Рассчитываем метрики
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

# Выводим метрики
print(f"Средняя абсолютная ошибка (MAE): {mae}")
print(f"Корень средней квадратичной ошибки (RMSE): {rmse}")
print(f"MAPE: {mape*100}%")
print(f"Коэффициент детерминации (R²): {r2}")

Средняя абсолютная ошибка (MAE): 2262353.4010323747
Корень средней квадратичной ошибки (RMSE): 2990379.6170987957
MAPE: 134.53757998759946%
Коэффициент детерминации (R²): 0.5473304853053769
